# Smart Tagging: events, tags and distributions

This notebook explores `BAR-ATH_tactical_smart_tagging.xml` locally. Each row represents an automatically generated tactical interval—not a possession label and not human ground truth.

The analysis covers event codes, teams, set-piece types, interval durations, missing tag values and timing across the match.

> Keep notebook outputs local and clear them before committing.

In [ ]:
from pathlib import Path
import os
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'games.py').exists() and (candidate / 'DATA.md').exists():
            return candidate
    raise FileNotFoundError('Could not find repository root')

ROOT = find_repo_root()
DATA_ROOT = Path(os.environ.get('MTAH_DATA_DIR', ROOT / 'data'))
FILENAME = 'BAR-ATH_tactical_smart_tagging.xml'
candidates = [DATA_ROOT / 'tactical' / FILENAME, DATA_ROOT / FILENAME]
PATH = next((p for p in candidates if p.exists()), None)
if PATH is None:
    raise FileNotFoundError(f'Could not find {FILENAME} in {candidates}')
FPS = 25
print('Reading:', PATH)

## 1. Convert the XML into a table

The XML stores one `instance` per interval. Tag groups vary by event type, so they are expanded into optional DataFrame columns.

In [ ]:
def parse_smart_tagging(path):
    rows = []
    root = ET.parse(path).getroot()
    for instance in root.iter('instance'):
        start_s = float(instance.findtext('start'))
        end_s = float(instance.findtext('end'))
        row = {
            'code': instance.findtext('code'),
            'start_s': start_s,
            'end_s': end_s,
            'duration_s': end_s - start_s,
            'start_frame': int(round(start_s * FPS)),
            'end_frame': int(round(end_s * FPS)),
        }
        for label in instance.findall('label'):
            group = label.findtext('group')
            value = label.findtext('text')
            if group:
                row[group] = value
        rows.append(row)
    return pd.DataFrame(rows).sort_values('start_s').reset_index(drop=True)

events = parse_smart_tagging(PATH)
print(f'{len(events):,} intervals; {len(events.columns)} columns')
display(events.head(12))

In [ ]:
schema = pd.DataFrame({
    'dtype': events.dtypes.astype(str),
    'non_null': events.notna().sum(),
    'missing_pct': (100 * events.isna().mean()).round(1),
    'unique_values': events.nunique(dropna=True),
})
display(schema)

## 2. Which tactical codes are present?

In [ ]:
code_counts = events['code'].value_counts().sort_values()
display(code_counts.rename('intervals').to_frame())

ax = code_counts.plot.barh(figsize=(10, 7), title='Smart Tagging intervals by code')
ax.set_xlabel('Intervals')
ax.set_ylabel('Code')
plt.tight_layout()
plt.show()

## 3. Team, half and tag-value distributions

Missing values are expected because not every tag group applies to every code. `N/A` is also a real tag value and must not silently become Home or Away.

In [ ]:
categorical_columns = [
    column for column in ['Team', 'Half', 'Type', 'Side',
                          'Direction of ball entry', 'Max Players in the box']
    if column in events.columns
]
for column in categorical_columns:
    print(f'\n{column}')
    display(events[column].value_counts(dropna=False).rename('count').to_frame())

In [ ]:
if 'Team' in events.columns:
    team_by_code = pd.crosstab(events['code'], events['Team'], dropna=False)
    display(team_by_code)
    team_by_code.plot.barh(stacked=True, figsize=(11, 8), title='Team tags within each tactical code')
    plt.xlabel('Intervals')
    plt.tight_layout()
    plt.show()

## 4. Interval-duration distributions

Smart Tagging intervals are windows, not exact possession or restart boundaries. Their duration distributions help show which codes describe long phases and which describe short events.

In [ ]:
duration_summary = events.groupby('code')['duration_s'].agg(
    intervals='size', minimum='min', median='median', mean='mean', maximum='max'
).sort_values('intervals', ascending=False)
display(duration_summary.round(2))

In [ ]:
ordered_codes = duration_summary.sort_values('median').index.tolist()
values = [events.loc[events['code'] == code, 'duration_s'].to_numpy() for code in ordered_codes]
fig, ax = plt.subplots(figsize=(11, 8))
ax.boxplot(values, labels=ordered_codes, vert=False, showfliers=False)
ax.set(title='Interval duration by tactical code', xlabel='Duration (seconds)', ylabel='Code')
fig.tight_layout()
plt.show()

## 5. Set pieces

Set-piece intervals are particularly useful as approximate restart evidence for the dead-ball state. They should not be treated as exact restart frames.

In [ ]:
set_pieces = events[events['code'].str.casefold() == 'set pieces'].copy()
columns = [c for c in ['Half', 'Team', 'Type', 'start_s', 'end_s', 'duration_s',
                         'start_frame', 'end_frame'] if c in set_pieces.columns]
display(set_pieces[columns].head(20))
if 'Type' in set_pieces.columns:
    display(set_pieces['Type'].value_counts(dropna=False).rename('count').to_frame())

## 6. When do the intervals occur?

In [ ]:
code_order = events['code'].value_counts().index.tolist()
code_to_y = {code: i for i, code in enumerate(code_order)}
fig, ax = plt.subplots(figsize=(14, 8))
for code, group in events.groupby('code'):
    y = code_to_y[code]
    ax.hlines(np.full(len(group), y), group['start_s'] / 60, group['end_s'] / 60, linewidth=2)
ax.set_yticks(range(len(code_order)), code_order)
ax.set(title='Smart Tagging timeline', xlabel='Minutes from tactical-video frame 0', ylabel='Code')
fig.tight_layout()
plt.show()

## What Person 1 should export

Convert intervals into frame-aligned indicator columns such as `set_piece_active`, `smart_tag_home`, `smart_tag_away`, and selected code flags. Preserve an explicit `N/A`/missing flag. Do not copy team tags directly into final possession labels: phases are approximate and can span different possession situations.

---

**Before committing:** restart the kernel, clear every output, and save.